# V20: Data Augmentation Maximalist — LOO + Mixup + 3-Tier PB

**Strategy:** Freeze architecture (V17 Blind MLP) + maximize data augmentation. NO architecture changes.

**Base:** V17 (Blind MLP + cosine_light + 3-tier PB, LB=3.63441 🏆 ALL-TIME BEST)
**Target:** LB > 4.0

### The 3 Boosters

| # | Booster | Type | Mechanism |
|---|---------|------|-----------|
| 1 | 3-Tier PseudoBulk | DATA (proven) | Bootstrap, per-channel, half-cell resampling |
| 2 | Genome-Wide LOO Synthetic | DATA (new) | Simulate KO for ALL 5127 genes from control cells |
| 3 | Cross-Gene Mixup | DATA (new) | Embedding-space interpolation between perturbations |

### Key Design Decisions
- **Architecture:** Strict V17 LightMLP (Embedding → 2-layer MLP → 5127 output). NO context vectors.
- **Loss:** `cosine_light` (λ=0.08, cos_right=0.0405). LOCKED from V16/V17.
- **LOO Rationale:** V17 only sees 80 gene_idx during training. LOO gives the model synthetic deltas for ALL genes, including the 60 test genes and 8 unknown scored genes.
- **Mixup Rationale:** Prevents memorization of 80 training deltas. Forces smooth embedding space.
- **GPU-accelerated:** LOO generation, normalization, and training all use PyTorch on GPU.

### Dead List (DO NOT re-introduce)
Context vectors, correlation matrices, MoE, ResidualMLP, perturbation conditioning, Replogle features, SVD denoising, global alpha, per-gene calibration, HybridMLP, consensus guidance

*V19.1 lesson: MORE pert-specific = WORSE LB. Proven across V13 (1.73), V19.1 (0.71). Blind MLP is king.*

*~By Mr.Mo & Mr.Jack*

In [ ]:
# ============================================================
# CELL 1: Setup & Drive Mount
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/Myllia_Challenge'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

for d in [BASE_DIR, DATA_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
print('Directories ready')

In [ ]:
# ============================================================
# CELL 2: Kaggle Download
# ============================================================
#@title Enter Kaggle API Token { display-mode: "form" }
KAGGLE_TOKEN = 'YOUR_KAGGLE_API_TOKEN_HERE'  #@param {type:"string"}
import shutil
if KAGGLE_TOKEN:
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(os.path.join(kaggle_dir, 'access_token'), 'w') as f: f.write(KAGGLE_TOKEN)
    os.chmod(os.path.join(kaggle_dir, 'access_token'), 0o600)
    !pip uninstall -y kaggle kagglehub kagglesdk 2>/dev/null
    !pip install -q --upgrade kaggle
    download_path = '/content/kaggle_data'
    os.makedirs(download_path, exist_ok=True)
    !kaggle competitions download -c echoes-of-silenced-genes -p {download_path}
    !unzip -q -o {download_path}/echoes-of-silenced-genes.zip -d {download_path}
    for f in os.listdir(download_path):
        src, dst = os.path.join(download_path, f), os.path.join(DATA_DIR, f)
        if os.path.isfile(src) and not os.path.exists(dst): shutil.copy2(src, dst)
    print(f'Data: {os.listdir(DATA_DIR)}')
else:
    print(f'Skip download. Files: {os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else "None"}')

In [ ]:
# ============================================================
# CELL 3: Dependencies (V20)
# ============================================================
!pip -q install scanpy anndata

import os, gc, json, math, shutil, warnings, re, datetime, time, copy
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import issparse
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader

from dataclasses import dataclass
from tqdm import tqdm
from sklearn.model_selection import KFold
from google.colab import files as colab_files

warnings.filterwarnings('ignore')
print(f'PyTorch {torch.__version__}, CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Dependencies ready (V20: +Dataset/DataLoader for LOO)')

In [ ]:
# ============================================================
# CELL 4: Config — V20 (Data Augmentation Maximalist)
#
# Architecture: FROZEN from V17 (LightMLP, Blind MLP)
# Loss: FROZEN from V16/V17 (cosine_light λ=0.08)
# NEW: LOO synthetic augmentation, embedding-space Mixup
# LOCKED: 3-tier PB, KO correction, seed formula
# ============================================================
USE_AMP = True
USE_TF32 = True

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = USE_TF32
    torch.backends.cudnn.allow_tf32 = USE_TF32
    torch.backends.cudnn.benchmark = True
    try: torch.set_float32_matmul_precision('high')
    except: pass

@dataclass
class Config:
    # Data
    NUM_GENES: int = 5127
    NUM_GENES_FULL: int = 19226
    NUM_TRAIN_PERTS: int = 80

    # PseudoBulk — 3-tier augmentation (LOCKED from V15)
    N_RESAMPLES_FULL: int = 10
    N_RESAMPLES_HALF: int = 5
    HALF_CELL_FRACTION: float = 0.5
    MIN_CELLS_PER_PERT: int = 10
    MIN_CELLS_PER_CHANNEL: int = 5
    CTRL_SUBSAMPLE: int = 200
    PB_QUALITY_THRESHOLD: float = 0.3

    # === BOOSTER 2: LOO Synthetic Perturbations ===
    LOO_PERCENTILE: float = 5.0       # bottom N% = "naturally low" cells
    LOO_MIN_CELLS: int = 10           # minimum cells in low-expression group
    LOO_N_RESAMPLES: int = 2          # bootstrap resamples per LOO gene (2 = ~10k samples)
    LOO_SAMPLE_WEIGHT: float = 0.05   # FIXED: REDUCED from 0.5 to 0.05 so 10k LOO samples don't drown out 80 GT

    # === BOOSTER 3: Cross-Gene Mixup ===
    MIXUP_P: float = 0.3              # probability of mixup per epoch
    MIXUP_BETA: float = 0.2           # Beta(0.2, 0.2) — bimodal, mostly near 0 or 1

    # MLP base (V6/V17 arch — LOCKED)
    EMBED_DIM: int = 64
    MLP_DEPTH: int = 2
    MLP_EPOCHS: int = 300
    MLP_LR: float = 5e-4
    EARLY_STOP: int = 50

    # V16/V17 winner HPs (LOCKED)
    MLP_HIDDEN: int = 384
    MLP_DROPOUT: float = 0.5
    WEIGHT_DECAY: float = 0.01
    COSINE_LAMBDA: float = 0.08
    COS_RIGHT: float = 0.0405
    GT_UPWEIGHT: float = 4.5
    N_MLP_ENSEMBLE: int = 30

    # Loss (LOCKED from V17)
    LOSS: str = 'cosine_light'

    # KO correction (V6 proven)
    KNOCKOUT_MULTIPLIER: float = 1.0

    # System
    N_FOLDS: int = 5
    SEED: int = 42
    DEVICE: str = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(cfg.SEED)

def clean_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'V20 Config: device={cfg.DEVICE}')
print(f'  Architecture: V17 LightMLP (FROZEN)')
print(f'  Loss: cosine_light λ={cfg.COSINE_LAMBDA}, cos_right={cfg.COS_RIGHT} (FROZEN)')
print(f'  gt_upweight={cfg.GT_UPWEIGHT}, N_ensemble={cfg.N_MLP_ENSEMBLE}')
print(f'  Booster 1: 3-Tier PB (quality_thresh={cfg.PB_QUALITY_THRESHOLD})')
print(f'  Booster 2: LOO (percentile={cfg.LOO_PERCENTILE}, min_cells={cfg.LOO_MIN_CELLS}, '
      f'resamples={cfg.LOO_N_RESAMPLES}, weight={cfg.LOO_SAMPLE_WEIGHT})')
print(f'  Booster 3: Mixup (p={cfg.MIXUP_P}, beta={cfg.MIXUP_BETA})')
print(f'  AMP={USE_AMP}, TF32={USE_TF32}')

In [ ]:
# ============================================================
# CELL 5: Load Competition Data (from V16 — unchanged)
# ============================================================
data_path = DATA_DIR if os.path.exists(DATA_DIR) and os.listdir(DATA_DIR) else '/content/kaggle_data'
print(f'Data path: {data_path}')

train_means = pd.read_csv(os.path.join(data_path, 'training_data_means.csv'))
first_col = train_means.iloc[:, 0].astype(str)
control_mask = first_col.str.lower().str.contains('non-targeting|control', na=False)
control_row = train_means[control_mask].iloc[0] if control_mask.any() else train_means.iloc[-1]
pert_df = train_means[~control_mask] if control_mask.any() else train_means.iloc[:-1]

gene_names = list(train_means.columns[1:])
control_expr = control_row.iloc[1:].values.astype(np.float32)
pert_names = pert_df.iloc[:, 0].tolist()
pert_expr = pert_df.iloc[:, 1:].values.astype(np.float32)
deltas = pert_expr - control_expr
gene_to_idx = {g: i for i, g in enumerate(gene_names)}
pert_to_idx = {p: i for i, p in enumerate(pert_names)}
pert_gene_indices = np.array([gene_to_idx.get(p, -1) for p in pert_names])

gt_df = pd.read_csv(os.path.join(data_path, 'training_data_ground_truth_table.csv'))
id_col = None
for c in ['pert_symbol', 'pert', 'gene', 'target_gene']:
    if c in gt_df.columns:
        id_col = c; break
if id_col:
    gt_df[id_col] = gt_df[id_col].astype(str)
    if set(pert_names).issubset(set(gt_df[id_col])):
        gt_df = gt_df.set_index(id_col).reindex(pert_names).reset_index()

weight_cols = [c for c in gt_df.columns if c.startswith('w_')]
weights_array = gt_df[weight_cols].values.astype(np.float32) if weight_cols else np.abs(deltas) + 0.1

sample_submission = pd.read_csv(os.path.join(data_path, 'sample_submission.csv'))
SUBMISSION_COLUMNS = list(sample_submission.columns)
val_pert_ids = sample_submission['pert_id'].tolist()
val_perts_df = pd.read_csv(os.path.join(data_path, 'pert_ids_val.csv'))

def _norm_pert_id(value):
    if value is None or (isinstance(value, float) and np.isnan(value)): return None
    s = str(value).strip()
    if not s: return None
    if s.startswith('pert_'): return s
    if s.isdigit(): return f'pert_{int(s)}'
    m = re.search(r'(\d+)', s)
    if m: return f'pert_{int(m.group(1))}'
    return s

val_perts_df['pert_id_norm'] = val_perts_df['pert_id'].apply(_norm_pert_id)
_sub_pert_id_norm = [_norm_pert_id(pid) for pid in val_pert_ids]

val_perts_known = val_perts_df
if 'class' in val_perts_df.columns:
    val_perts_known = val_perts_df[val_perts_df['class'].astype(str).str.lower().str.contains('val')]
pert_id_to_symbol = dict(zip(val_perts_known['pert_id_norm'], val_perts_known['pert']))
val_pert_symbols = [pert_id_to_symbol.get(pid) for pid in _sub_pert_id_norm]

gene_names_upper = {g.upper(): g for g in gene_names}
val_gene_indices = []
for symbol in val_pert_symbols:
    if symbol is not None and symbol in gene_to_idx:
        val_gene_indices.append(gene_to_idx[symbol])
    elif symbol is not None and symbol.upper() in gene_names_upper:
        val_gene_indices.append(gene_to_idx[gene_names_upper[symbol.upper()]])
    else:
        val_gene_indices.append(-1)

valid_idx_count = sum(1 for i in val_gene_indices if i >= 0)
print(f'Genes: {len(gene_names)}, Train perts: {len(pert_names)}, Val perts: {len(val_pert_symbols)}')
print(f'Val gene idx valid: {valid_idx_count}/{len(val_pert_symbols)}')
print(f'Deltas shape: {deltas.shape}, range [{deltas.min():.3f}, {deltas.max():.3f}]')
print(f'Weight cols: {len(weight_cols)}, weights shape: {weights_array.shape}')

In [ ]:
# ============================================================
# CELL 6: Competition Metric (exact, cos_right=0.2) + KO Stats
# (from V16 — unchanged)
# ============================================================
baseline_pred = np.mean(deltas, axis=0)

baseline_wmae = None
if 'baseline_wmae' in gt_df.columns:
    baseline_wmae = gt_df['baseline_wmae'].values.astype(np.float64)
if baseline_wmae is None:
    baseline_wmae = np.mean(np.abs(deltas - baseline_pred) * weights_array, axis=1)

class CompetitionMetric:
    def __init__(self, eps=1e-12, max_log2=5.0, left=0.0, right=0.2):
        self.eps, self.max_log2, self.left, self.right = eps, max_log2, left, right

    def smoothstep(self, t):
        return t * t * (3.0 - 2.0 * t)

    def gate_smoothstep(self, x):
        t = np.clip((x - self.left) / (self.right - self.left), 0.0, 1.0)
        return self.smoothstep(t)

    def weighted_cosine(self, a, b):
        a, b = np.asarray(a, dtype=np.float64).ravel(), np.asarray(b, dtype=np.float64).ravel()
        x = np.maximum(np.abs(a), np.abs(b))
        w2 = self.gate_smoothstep(x) ** 2
        num = np.sum(w2 * a * b)
        den = np.sqrt(np.sum(w2 * a * a)) * np.sqrt(np.sum(w2 * b * b))
        return float(num / den) if den > self.eps else 0.0

    def score(self, preds, targets, weights, bl_wmae):
        y_true = np.asarray(targets, dtype=np.float64)
        y_pred = np.asarray(preds, dtype=np.float64)
        w = np.asarray(weights, dtype=np.float64)
        base = np.asarray(bl_wmae, dtype=np.float64).ravel()
        if base.shape[0] != y_true.shape[0]:
            base = np.mean(np.abs(y_true - baseline_pred) * w, axis=1)
        pred_wmae = np.maximum(np.mean(np.abs(y_true - y_pred) * w, axis=1), self.eps)
        base = np.maximum(base, self.eps)
        terms = np.minimum(np.log2(base / pred_wmae), self.max_log2)
        sum_wmae = float(np.sum(terms))
        wcos = self.weighted_cosine(y_pred.ravel(), y_true.ravel())
        return round(float(sum_wmae * max(0.0, wcos)), 5), sum_wmae, wcos

    def score_per_pert(self, preds, targets, weights, bl_wmae):
        y_true = np.asarray(targets, dtype=np.float64)
        y_pred = np.asarray(preds, dtype=np.float64)
        w = np.asarray(weights, dtype=np.float64)
        base = np.asarray(bl_wmae, dtype=np.float64).ravel()
        pred_wmae = np.maximum(np.mean(np.abs(y_true - y_pred) * w, axis=1), self.eps)
        base = np.maximum(base, self.eps)
        terms = np.minimum(np.log2(base / pred_wmae), self.max_log2)
        wcos = self.weighted_cosine(y_pred.ravel(), y_true.ravel())
        return terms, wcos

metric = CompetitionMetric()

ko_effects = [deltas[i, idx] for i, idx in enumerate(pert_gene_indices) if idx >= 0]
knockout_stats = {'mean': np.mean(ko_effects), 'std': np.std(ko_effects)}
print(f'baseline_wmae: mean={baseline_wmae.mean():.4f}, std={baseline_wmae.std():.4f}')
print(f'Knockout mean: {knockout_stats["mean"]:.4f}, std: {knockout_stats["std"]:.4f}')

def apply_ko(pred, idx, stats, mult=1.0):
    pred = pred.copy()
    if 0 <= idx < len(pred):
        pred[idx] = stats['mean'] * mult
    return pred

bl_score, bl_w, bl_wcos = metric.score(
    np.tile(baseline_pred, (len(pert_names), 1)), deltas, weights_array, baseline_wmae)
print(f'Baseline (mean) score: {bl_score:.4f} (W={bl_w:.4f}, wcos={bl_wcos:.4f}) -- expected ~0')

In [ ]:
# ============================================================
# CELL 7: 3-Tier PseudoBulk Generation (from V16 — unchanged)
# ============================================================
import scanpy as sc

print('=' * 70)
print('CELL 7: 3-Tier PseudoBulk Generation')
print('=' * 70)

h5ad_path = os.path.join(data_path, 'training_cells.h5ad')
assert os.path.exists(h5ad_path), f'training_cells.h5ad not found at {h5ad_path}'

adata_train = sc.read_h5ad(h5ad_path)
print(f'Shape: {adata_train.shape[0]} cells x {adata_train.shape[1]} genes')

X_check = adata_train.X[:200]
if sp.issparse(X_check): X_check = X_check.toarray()
data_max = X_check.max()
IS_RAW = data_max > 50
print(f'Data max (first 200 cells): {data_max:.1f} -> {"Raw UMI" if IS_RAW else "Already normalized"}')

h5_genes = list(adata_train.var_names)
h5_gene_upper = {g.upper(): i for i, g in enumerate(h5_genes)}
gene_subset_idx = []
comp_gene_order = []
for ci, g in enumerate(gene_names):
    gu = g.upper()
    if gu in h5_gene_upper:
        gene_subset_idx.append(h5_gene_upper[gu])
        comp_gene_order.append(ci)
gene_subset_idx = np.array(gene_subset_idx)
comp_gene_order = np.array(comp_gene_order)
print(f'Gene overlap: {len(gene_subset_idx)}/{cfg.NUM_GENES} ({100*len(gene_subset_idx)/cfg.NUM_GENES:.1f}%)')

CANDIDATE_COLS = ['pert_symbol', 'perturbation', 'pert', 'gene', 'target_gene',
                  'perturbation_name', 'gene_symbol', 'gene_name', 'symbol']

def _count_pert_matches(col_series, pert_names_list):
    col_upper = set(col_series.astype(str).str.upper().unique())
    return sum(1 for p in pert_names_list if p.upper() in col_upper)

pert_col_h5 = None
pert_col_matches = 0
_sgrna_to_gene = {}

for c in CANDIDATE_COLS:
    if c in adata_train.obs.columns:
        n_matches = _count_pert_matches(adata_train.obs[c], pert_names)
        print(f'  Column "{c}": {n_matches}/{len(pert_names)} matches')
        if n_matches > pert_col_matches:
            pert_col_h5, pert_col_matches = c, n_matches

if pert_col_matches < len(pert_names) // 2:
    for c in adata_train.obs.columns:
        if c == pert_col_h5: continue
        col = adata_train.obs[c]
        if col.dtype == object or str(col.dtype) == 'category':
            if 5 < col.nunique() < 50000:
                n_matches = _count_pert_matches(col, pert_names)
                if n_matches > pert_col_matches:
                    print(f'  Better column "{c}": {n_matches}/{len(pert_names)} matches')
                    pert_col_h5, pert_col_matches = c, n_matches

if pert_col_matches < len(pert_names) // 2:
    print(f'  Trying substring extraction...')
    pert_names_upper = {p.upper(): p for p in pert_names}
    for c in adata_train.obs.columns:
        col = adata_train.obs[c]
        if col.dtype != object and str(col.dtype) != 'category': continue
        extraction_map = {}
        for val in col.astype(str).unique():
            for delim in ['_', '-', '.', '|', ':']:
                for part in val.upper().split(delim):
                    part = part.strip()
                    if part in pert_names_upper:
                        extraction_map[val] = pert_names_upper[part]
                        break
                if val in extraction_map: break
        if len(extraction_map) > pert_col_matches:
            print(f'  Column "{c}": extracted {len(extraction_map)} gene names')
            pert_col_h5, pert_col_matches = c, len(extraction_map)
            _sgrna_to_gene = extraction_map

assert pert_col_h5 is not None, f'No perturbation column found!'
print(f'\nPerturbation column: "{pert_col_h5}" ({pert_col_matches}/{len(pert_names)} matches)')

pert_str_h5 = adata_train.obs[pert_col_h5].astype(str)
ctrl_mask_h5 = pert_str_h5.str.lower().str.contains('non.targeting|control|ctrl', regex=True, na=False)
if 'is_control' in adata_train.obs.columns:
    ctrl_mask_h5 = ctrl_mask_h5 | adata_train.obs['is_control'].astype(bool)
if 'control' in adata_train.obs.columns and adata_train.obs['control'].dtype == bool:
    ctrl_mask_h5 = ctrl_mask_h5 | adata_train.obs['control']
ctrl_cell_idx = np.where(ctrl_mask_h5.values)[0]
print(f'Control cells: {len(ctrl_cell_idx)}')

channel_col = None
for c in ['channel', 'batch', 'lane', 'pool', 'sample']:
    if c in adata_train.obs.columns:
        n_unique = adata_train.obs[c].nunique()
        if 2 <= n_unique <= 20:
            channel_col = c; break
if channel_col is None:
    for c in adata_train.obs.columns:
        col = adata_train.obs[c]
        if col.dtype == object or str(col.dtype) == 'category':
            if 2 <= col.nunique() <= 20 and c not in [pert_col_h5]:
                channel_col = c; break

channel_names = []
if channel_col:
    channel_names = sorted(adata_train.obs[channel_col].astype(str).unique())
    print(f'Channel column: "{channel_col}" ({len(channel_names)} channels)')
else:
    print(f'WARNING: No channel column found. Tier 2 SKIPPED.')

cells_per_pert = {}
if _sgrna_to_gene:
    gene_to_sgrna_ids = {}
    for sgrna_val, gene_name in _sgrna_to_gene.items():
        gene_to_sgrna_ids.setdefault(gene_name, []).append(sgrna_val)
    for p in pert_names:
        if p in gene_to_sgrna_ids:
            all_idx = []
            for sgrna_val in gene_to_sgrna_ids[p]:
                mask = (pert_str_h5 == sgrna_val) & (~ctrl_mask_h5)
                all_idx.extend(np.where(mask.values)[0].tolist())
            if len(all_idx) >= 3:
                cells_per_pert[p] = np.array(all_idx)

for p in pert_names:
    if p in cells_per_pert: continue
    mask = (pert_str_h5.str.upper() == p.upper()) & (~ctrl_mask_h5)
    cell_idx = np.where(mask.values)[0]
    if len(cell_idx) >= 3:
        cells_per_pert[p] = cell_idx

print(f'Perturbations with cells: {len(cells_per_pert)}/{len(pert_names)}')

cells_per_pert_channel = {}
ctrl_cells_per_channel = {}
if channel_col:
    obs_ch = adata_train.obs[channel_col].astype(str).values
    for ch in channel_names:
        ch_mask = obs_ch == ch
        ctrl_ch_idx = np.where(ch_mask & ctrl_mask_h5.values)[0]
        if len(ctrl_ch_idx) > 0:
            ctrl_cells_per_channel[ch] = ctrl_ch_idx
    for p, cell_idx in cells_per_pert.items():
        for ch in channel_names:
            ch_mask_p = obs_ch[cell_idx] == ch
            ch_idx = cell_idx[ch_mask_p]
            if len(ch_idx) >= cfg.MIN_CELLS_PER_CHANNEL:
                cells_per_pert_channel[(p, ch)] = ch_idx

print(f'\nNormalizing and caching...')
t0_cache = time.time()

def normalize_cells_to_dense(X_raw, gene_subset_idx, is_raw=True):
    if sp.issparse(X_raw): X_raw = X_raw.toarray()
    X_raw = X_raw.astype(np.float64)
    if is_raw:
        totals = X_raw.sum(axis=1, keepdims=True)
        totals[totals == 0] = 1
        normalized_full = np.log2(1.0 + X_raw / totals * 10000.0)
    else:
        normalized_full = X_raw
    return normalized_full[:, gene_subset_idx].astype(np.float32)

ctrl_dense = normalize_cells_to_dense(adata_train.X[ctrl_cell_idx], gene_subset_idx, is_raw=IS_RAW)
ctrl_dense_per_channel = {}
if channel_col:
    for ch, ch_idx in ctrl_cells_per_channel.items():
        ctrl_dense_per_channel[ch] = normalize_cells_to_dense(
            adata_train.X[ch_idx], gene_subset_idx, is_raw=IS_RAW)

pert_cells_dense = {}
for p, cell_idx in tqdm(cells_per_pert.items(), desc='Caching pert cells'):
    pert_cells_dense[p] = normalize_cells_to_dense(
        adata_train.X[cell_idx], gene_subset_idx, is_raw=IS_RAW)

pert_cells_dense_channel = {}
if channel_col:
    for (p, ch), ch_idx in tqdm(cells_per_pert_channel.items(), desc='Caching pert-channel'):
        pert_cells_dense_channel[(p, ch)] = normalize_cells_to_dense(
            adata_train.X[ch_idx], gene_subset_idx, is_raw=IS_RAW)

print(f'  Caching done in {time.time()-t0_cache:.1f}s')
del adata_train; clean_memory()

print(f'\nGenerating 3-Tier PseudoBulk...')
rng = np.random.RandomState(cfg.SEED)
n_ctrl = ctrl_dense.shape[0]
n_ctrl_sub = min(n_ctrl, cfg.CTRL_SUBSAMPLE)

pb_deltas_list, pb_pert_indices_list, pb_pert_names_list = [], [], []
pb_tiers_list, pb_n_cells_list = [], []

# Tier 1: Full Bootstrap
n_t1 = 0
for p in pert_names:
    if p not in pert_cells_dense: continue
    pert_cells = pert_cells_dense[p]
    n_pert = pert_cells.shape[0]
    if n_pert < cfg.MIN_CELLS_PER_PERT: continue
    gi = pert_gene_indices[pert_to_idx[p]]
    for _ in range(cfg.N_RESAMPLES_FULL):
        pert_idx = rng.randint(0, n_pert, size=n_pert)
        ctrl_idx = rng.randint(0, n_ctrl, size=n_ctrl_sub)
        delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
        delta[comp_gene_order] = pert_cells[pert_idx].mean(axis=0) - ctrl_dense[ctrl_idx].mean(axis=0)
        pb_deltas_list.append(delta)
        pb_pert_indices_list.append(gi)
        pb_pert_names_list.append(p)
        pb_tiers_list.append(1); pb_n_cells_list.append(n_pert); n_t1 += 1
print(f'  Tier 1: {n_t1}')

# Tier 2: Per-Channel
n_t2 = 0
if channel_col and ctrl_dense_per_channel:
    for p in pert_names:
        gi = pert_gene_indices[pert_to_idx[p]]
        for ch in channel_names:
            if (p, ch) not in pert_cells_dense_channel: continue
            if ch not in ctrl_dense_per_channel: continue
            pert_ch = pert_cells_dense_channel[(p, ch)]
            ctrl_ch = ctrl_dense_per_channel[ch]
            delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
            delta[comp_gene_order] = pert_ch.mean(axis=0) - ctrl_ch.mean(axis=0)
            pb_deltas_list.append(delta)
            pb_pert_indices_list.append(gi)
            pb_pert_names_list.append(p)
            pb_tiers_list.append(2); pb_n_cells_list.append(pert_ch.shape[0]); n_t2 += 1
print(f'  Tier 2: {n_t2}')

# Tier 3: Half-Cell
n_t3 = 0
for p in pert_names:
    if p not in pert_cells_dense: continue
    pert_cells = pert_cells_dense[p]
    n_pert = pert_cells.shape[0]
    if n_pert < cfg.MIN_CELLS_PER_PERT: continue
    gi = pert_gene_indices[pert_to_idx[p]]
    half_n = max(3, int(n_pert * cfg.HALF_CELL_FRACTION))
    for _ in range(cfg.N_RESAMPLES_HALF):
        pert_idx = rng.choice(n_pert, size=half_n, replace=False)
        ctrl_idx = rng.randint(0, n_ctrl, size=n_ctrl_sub)
        delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
        delta[comp_gene_order] = pert_cells[pert_idx].mean(axis=0) - ctrl_dense[ctrl_idx].mean(axis=0)
        pb_deltas_list.append(delta)
        pb_pert_indices_list.append(gi)
        pb_pert_names_list.append(p)
        pb_tiers_list.append(3); pb_n_cells_list.append(half_n); n_t3 += 1
print(f'  Tier 3: {n_t3}')

# Quality Gate
pb_deltas_raw = np.array(pb_deltas_list, dtype=np.float32)
pb_pert_indices_raw = np.array(pb_pert_indices_list)
pb_pert_names_raw = pb_pert_names_list
pb_tiers_raw = np.array(pb_tiers_list)
pb_n_cells_raw = np.array(pb_n_cells_list)

keep_mask = np.ones(len(pb_deltas_raw), dtype=bool)
for i in range(len(pb_deltas_raw)):
    pn = pb_pert_names_raw[i]
    corr = np.corrcoef(pb_deltas_raw[i], deltas[pert_to_idx[pn]])[0, 1]
    if np.isnan(corr) or corr < cfg.PB_QUALITY_THRESHOLD:
        keep_mask[i] = False

print(f'  Quality gate: kept {keep_mask.sum()}/{len(pb_deltas_raw)}')

pb_deltas = pb_deltas_raw[keep_mask]
pb_pert_indices = pb_pert_indices_raw[keep_mask]
pb_pert_names = [pb_pert_names_raw[i] for i in range(len(pb_pert_names_raw)) if keep_mask[i]]
pb_tiers = pb_tiers_raw[keep_mask]
pb_n_cells = pb_n_cells_raw[keep_mask]

pb_cell_weights = np.sqrt(pb_n_cells.astype(np.float32))
mean_sqrt = pb_cell_weights.mean()
if mean_sqrt > 0: pb_cell_weights /= mean_sqrt

pb_weights = np.zeros((len(pb_deltas), cfg.NUM_GENES), dtype=np.float32)
for i, pn in enumerate(pb_pert_names):
    pb_weights[i] = weights_array[pert_to_idx[pn]]

del pert_cells_dense, ctrl_dense, pert_cells_dense_channel, ctrl_dense_per_channel
clean_memory()

n_pb_perts = len(set(pb_pert_names))
print(f'\nPB ready: {n_pb_perts} perts, {len(pb_deltas)} samples')
print(f'Total: {len(pb_deltas)} PB + {len(pert_names)} GT = {len(pb_deltas) + len(pert_names)}')

In [ ]:
# ============================================================
# CELL 8: LOO Synthetic Perturbation Generator (GPU-Accelerated)
#
# For each of the 5127 competition genes, simulate a knockout:
#   1. Find control cells where gene G is in the bottom 5th percentile
#   2. Average these "naturally low G" cells
#   3. Subtract the global control mean
#   4. Result: synthetic delta for gene G's knockout
#
# GPU-accelerated: all operations on torch tensors.
# Produces loo_deltas (N_loo x 5127), loo_gene_indices (N_loo,)
# ============================================================
print('=' * 70)
print('CELL 8: LOO Synthetic Perturbation Generator (GPU)')
print('=' * 70)

# We need ctrl_dense which was deleted in Cell 7. Reload just control cells.
import scanpy as sc
h5ad_path = os.path.join(data_path, 'training_cells.h5ad')
adata_loo = sc.read_h5ad(h5ad_path)

# Re-derive control cell indices
pert_str_loo = adata_loo.obs[pert_col_h5].astype(str)
ctrl_mask_loo = pert_str_loo.str.lower().str.contains('non.targeting|control|ctrl', regex=True, na=False)
if 'is_control' in adata_loo.obs.columns:
    ctrl_mask_loo = ctrl_mask_loo | adata_loo.obs['is_control'].astype(bool)
if 'control' in adata_loo.obs.columns and adata_loo.obs['control'].dtype == bool:
    ctrl_mask_loo = ctrl_mask_loo | adata_loo.obs['control']
ctrl_idx_loo = np.where(ctrl_mask_loo.values)[0]

print(f'Reloaded {len(ctrl_idx_loo)} control cells for LOO generation')

# Normalize control cells (all 19226 genes first, then subset)
X_ctrl_raw = adata_loo.X[ctrl_idx_loo]
if sp.issparse(X_ctrl_raw): X_ctrl_raw = X_ctrl_raw.toarray()
X_ctrl_raw = X_ctrl_raw.astype(np.float64)
if IS_RAW:
    totals = X_ctrl_raw.sum(axis=1, keepdims=True)
    totals[totals == 0] = 1
    X_ctrl_norm = np.log2(1.0 + X_ctrl_raw / totals * 10000.0)
else:
    X_ctrl_norm = X_ctrl_raw
# Subset to competition genes (using gene_subset_idx from Cell 7)
ctrl_subset = X_ctrl_norm[:, gene_subset_idx].astype(np.float32)
del X_ctrl_raw, X_ctrl_norm, adata_loo; clean_memory()

n_ctrl_cells = ctrl_subset.shape[0]
n_overlap_genes = ctrl_subset.shape[1]  # = len(comp_gene_order)
print(f'Control cells: {n_ctrl_cells} x {n_overlap_genes} (subset genes)')

# Move to GPU for fast percentile computation
t0_loo = time.time()
device = cfg.DEVICE
ctrl_gpu = torch.tensor(ctrl_subset, dtype=torch.float32, device=device)
ctrl_mean_gpu = ctrl_gpu.mean(dim=0)  # global control mean (n_overlap_genes,)

# Build mapping: competition gene index -> column in ctrl_subset
# comp_gene_order[j] is the competition gene index for column j in ctrl_subset
comp_gene_to_col = {}
for col_j, comp_gi in enumerate(comp_gene_order):
    comp_gene_to_col[int(comp_gi)] = col_j

# Generate LOO deltas for ALL competition genes that have a column in ctrl_subset
rng_loo = np.random.RandomState(cfg.SEED + 999)
loo_deltas_list = []
loo_gene_indices_list = []
loo_gene_names_list = []
n_skipped = 0

percentile_threshold = cfg.LOO_PERCENTILE

for comp_gi in range(cfg.NUM_GENES):
    if comp_gi not in comp_gene_to_col:
        n_skipped += 1
        continue

    col_j = comp_gene_to_col[comp_gi]
    gene_expr = ctrl_gpu[:, col_j]  # expression of this gene across all control cells

    # Find cells where this gene is in bottom percentile (naturally low)
    threshold = torch.quantile(gene_expr, percentile_threshold / 100.0)
    low_mask = gene_expr <= threshold

    n_low = low_mask.sum().item()
    if n_low < cfg.LOO_MIN_CELLS:
        n_skipped += 1
        continue

    low_cells = ctrl_gpu[low_mask]  # cells where gene G is naturally low

    # Generate bootstrap resamples
    for _ in range(cfg.LOO_N_RESAMPLES):
        # Bootstrap resample from low-expression cells
        boot_idx = torch.randint(0, n_low, (n_low,), device=device)
        low_mean = low_cells[boot_idx].mean(dim=0)

        # Bootstrap resample from all control cells for the "normal" baseline
        ctrl_boot_idx = torch.randint(0, n_ctrl_cells,
                                       (min(n_ctrl_cells, cfg.CTRL_SUBSAMPLE),), device=device)
        ctrl_boot_mean = ctrl_gpu[ctrl_boot_idx].mean(dim=0)

        # Synthetic delta = low_mean - ctrl_mean
        syn_delta_subset = (low_mean - ctrl_boot_mean).cpu().numpy()

        # Map back to full 5127-dim competition gene space
        full_delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
        full_delta[comp_gene_order] = syn_delta_subset

        loo_deltas_list.append(full_delta)
        loo_gene_indices_list.append(comp_gi)

    loo_gene_names_list.append(gene_names[comp_gi])

del ctrl_gpu; clean_memory()

loo_deltas = np.array(loo_deltas_list, dtype=np.float32)
loo_gene_indices = np.array(loo_gene_indices_list, dtype=np.int64)
n_loo_genes = len(loo_gene_names_list)

elapsed_loo = time.time() - t0_loo
print(f'\nLOO generation done in {elapsed_loo:.1f}s')
print(f'  LOO genes generated: {n_loo_genes}/{cfg.NUM_GENES} (skipped {n_skipped})')
print(f'  LOO samples: {len(loo_deltas)} ({cfg.LOO_N_RESAMPLES} resamples/gene)')
print(f'  LOO deltas range: [{loo_deltas.min():.4f}, {loo_deltas.max():.4f}]')
print(f'  LOO mean |delta|: {np.abs(loo_deltas).mean():.6f}')

# Check coverage of val (test) genes
val_genes_covered = sum(1 for gi in val_gene_indices if gi >= 0 and gi in comp_gene_to_col)
val_genes_in_loo = sum(1 for gi in val_gene_indices
                       if gi >= 0 and any(loo_gene_indices == gi))
print(f'\n  Val gene coverage: {val_genes_covered}/{len(val_gene_indices)} in ctrl_subset')
print(f'  Val genes with LOO data: {val_genes_in_loo}/{len(val_gene_indices)}')

# Check overlap with training genes
train_in_loo = sum(1 for gi in pert_gene_indices if gi >= 0 and any(loo_gene_indices == gi))
print(f'  Training genes with LOO data: {train_in_loo}/{len(pert_names)}')

# Build LOO weights (uniform, scaled by LOO_SAMPLE_WEIGHT)
loo_sample_weights = np.full(len(loo_deltas), cfg.LOO_SAMPLE_WEIGHT, dtype=np.float32)

# Use average competition weights for LOO samples (no per-pert weights available)
avg_weights = weights_array.mean(axis=0)
loo_weights = np.tile(avg_weights, (len(loo_deltas), 1))

# Keep ctrl_subset in RAM for potential reuse
ctrl_dense_for_loo = ctrl_subset
del ctrl_subset

print(f'\nLOO ready: {n_loo_genes} genes, {len(loo_deltas)} samples')
print(f'Total data: {len(pb_deltas)} PB + {len(pert_names)} GT + {len(loo_deltas)} LOO = '
      f'{len(pb_deltas) + len(pert_names) + len(loo_deltas)}')
print('=' * 70)

In [ ]:
# ============================================================
# CELL 9: Model + Loss + Training Function (V20)
#
# Model: V17 LightMLP (FROZEN architecture)
# Loss: cosine_light (FROZEN from V16/V17)
# NEW: Embedding-space Mixup (Booster 3)
# NEW: LOO data integration in training loop
# ============================================================
print('=' * 70)
print('CELL 9: V20 Model + Loss + Training')
print('=' * 70)

# ==================== LOSS FUNCTION ====================

def cosine_light_loss(pred, target, weights, sample_weights=None,
                      lam=None, cos_right=None):
    if lam is None: lam = cfg.COSINE_LAMBDA
    if cos_right is None: cos_right = cfg.COS_RIGHT
    per_sample = (weights * torch.abs(pred - target)).mean(dim=1)
    if sample_weights is not None:
        per_sample = per_sample * sample_weights
    w_mae = per_sample.mean()
    x = torch.maximum(pred.abs(), target.abs())
    t = (x / cos_right).clamp(0, 1)
    sw = t * t * (3 - 2 * t)
    cos = F.cosine_similarity(pred * sw, target * sw, dim=-1).mean()
    return w_mae - lam * cos

# ==================== MODEL (V17 LightMLP — FROZEN) ====================

class LightMLP(nn.Module):
    def __init__(self, n_genes, hidden=256, dropout=0.3, depth=2, embed_dim=64):
        super().__init__()
        self.n_genes = n_genes
        self.gene_embed = nn.Embedding(n_genes + 1, embed_dim)
        layers = []
        in_dim = embed_dim
        for _ in range(depth):
            layers.extend([nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden),
                          nn.ReLU(), nn.Dropout(dropout)])
            in_dim = hidden
        layers.append(nn.Linear(in_dim, n_genes))
        self.mlp = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
        nn.init.normal_(self.gene_embed.weight, std=0.02)

    def forward(self, gene_idx, mixup_embed=None):
        if mixup_embed is not None:
            return self.mlp(mixup_embed)
        safe_idx = gene_idx.clone()
        safe_idx[safe_idx < 0] = self.n_genes
        safe_idx[safe_idx >= self.n_genes] = self.n_genes
        return self.mlp(self.gene_embed(safe_idx))

    def get_embed(self, gene_idx):
        safe_idx = gene_idx.clone()
        safe_idx[safe_idx < 0] = self.n_genes
        safe_idx[safe_idx >= self.n_genes] = self.n_genes
        return self.gene_embed(safe_idx)

# ==================== TRAINING FUNCTION ====================

def train_v20(model, pb_indices, pb_deltas_arr, pb_weights_arr, pb_sample_w,
              gt_indices, gt_deltas_arr, gt_weights_arr, gt_upweight,
              loo_indices, loo_deltas_arr, loo_weights_arr, loo_sample_w,
              val_indices, val_deltas, val_weights, val_bl_wmae,
              use_mixup=True, mixup_p=0.3, mixup_beta=0.2,
              weight_decay=None, epochs=None, lr=None, patience=None,
              device='cuda'):
    """V20 training: GT + PB + LOO data with embedding-space Mixup."""
    epochs = epochs or cfg.MLP_EPOCHS
    lr = lr or cfg.MLP_LR
    patience = patience or cfg.EARLY_STOP
    wd = weight_decay if weight_decay is not None else cfg.WEIGHT_DECAY

    model = model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 100)

    best_score, best_state, pat_count = -float('inf'), None, 0
    amp_on = USE_AMP and str(device).startswith('cuda')
    scaler = torch.cuda.amp.GradScaler(enabled=amp_on)

    # Combine PB + GT + LOO data
    n_pb = len(pb_indices)
    n_gt = len(gt_indices)
    n_loo = len(loo_indices)

    all_i = np.concatenate([pb_indices, gt_indices, loo_indices])
    all_d = np.concatenate([pb_deltas_arr, gt_deltas_arr, loo_deltas_arr], axis=0).astype(np.float32)
    all_w = np.concatenate([pb_weights_arr, gt_weights_arr, loo_weights_arr], axis=0).astype(np.float32)
    all_sw = np.concatenate([
        pb_sample_w.astype(np.float32) if len(pb_sample_w) > 0 else np.array([], dtype=np.float32),
        np.full(n_gt, gt_upweight, dtype=np.float32),
        loo_sample_w.astype(np.float32) if len(loo_sample_w) > 0 else np.array([], dtype=np.float32),
    ])

    va_i = torch.tensor(val_indices, dtype=torch.long, device=device)

    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(len(all_i))
        tr_i = torch.tensor(all_i[perm], dtype=torch.long, device=device)
        tr_d = torch.tensor(all_d[perm], dtype=torch.float32, device=device)
        tr_w = torch.tensor(all_w[perm], dtype=torch.float32, device=device)
        tr_sw = torch.tensor(all_sw[perm], dtype=torch.float32, device=device)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp_on):
            # Booster 3: Embedding-space Mixup
            if use_mixup and np.random.rand() < mixup_p and len(perm) > 1:
                alpha = np.random.beta(mixup_beta, mixup_beta)
                partner = torch.randperm(len(perm), device=device)
                # Mix embeddings (not just targets)
                embed_a = model.get_embed(tr_i)
                embed_b = model.get_embed(tr_i[partner])
                mixed_embed = alpha * embed_a + (1 - alpha) * embed_b
                tr_d = alpha * tr_d + (1 - alpha) * tr_d[partner]
                tr_w = alpha * tr_w + (1 - alpha) * tr_w[partner]
                tr_sw = alpha * tr_sw + (1 - alpha) * tr_sw[partner]
                pred = model(tr_i, mixup_embed=mixed_embed)
            else:
                pred = model(tr_i)

            loss = cosine_light_loss(pred, tr_d, tr_w, tr_sw)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        scheduler.step()

        del tr_i, tr_d, tr_w, tr_sw

        # Validation every 10 epochs
        if (ep + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=amp_on):
                    vp = model(va_i).float().cpu().numpy()
            s, _, _ = metric.score(vp, val_deltas, val_weights, val_bl_wmae)
            if s > best_score:
                best_score = s
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                pat_count = 0
            else:
                pat_count += 10
            if pat_count >= patience:
                break

    if best_state: model.load_state_dict(best_state)

    del va_i; clean_memory()
    return model.cpu(), best_score


def get_fold_pb(val_pert_set):
    """Return PB samples excluding val fold perturbations."""
    mask = np.array([pn not in val_pert_set for pn in pb_pert_names])
    if mask.any():
        assert len(set(np.array(pb_pert_names)[mask]) & val_pert_set) == 0, 'LEAKAGE!'
    if mask.sum() == 0:
        e = np.float32
        return (np.array([], dtype=np.int64), np.zeros((0, cfg.NUM_GENES), dtype=e),
                np.zeros((0, cfg.NUM_GENES), dtype=e), np.array([], dtype=e))
    return (pb_pert_indices[mask], pb_deltas[mask], pb_weights[mask],
            pb_cell_weights[mask])


def get_fold_loo(val_pert_gene_set):
    """Return LOO samples excluding val fold gene indices (prevent leakage)."""
    mask = np.array([gi not in val_pert_gene_set for gi in loo_gene_indices])
    if mask.sum() == 0:
        e = np.float32
        return (np.array([], dtype=np.int64), np.zeros((0, cfg.NUM_GENES), dtype=e),
                np.zeros((0, cfg.NUM_GENES), dtype=e), np.array([], dtype=e))
    return (loo_gene_indices[mask], loo_deltas[mask], loo_weights[mask],
            loo_sample_weights[mask])


# --- Model summary ---
test_model = LightMLP(cfg.NUM_GENES, hidden=cfg.MLP_HIDDEN,
                      dropout=cfg.MLP_DROPOUT, depth=cfg.MLP_DEPTH,
                      embed_dim=cfg.EMBED_DIM)
n_params = sum(p.numel() for p in test_model.parameters())
print(f'\nLightMLP params: {n_params:,}')
del test_model

print(f'Loss: cosine_light (λ={cfg.COSINE_LAMBDA}, cos_right={cfg.COS_RIGHT})')
print(f'Training: train_v20() with GT + PB + LOO + Mixup')
print('=' * 70)

In [ ]:
# ============================================================
# CELL 10: 5-Fold CV (V20)
#
# Compare V17 baseline vs Mixup vs LOO vs Both
# ============================================================
print('=' * 70)
print('CELL 10: 5-FOLD CV — V17 BASELINE vs V20 (LOO + MIXUP)')
print('=' * 70)

kf = KFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.SEED)
all_splits = list(kf.split(range(len(pert_names))))

configs_to_test = [
    {'name': 'V17_baseline', 'use_loo': False, 'use_mixup': False},
    {'name': 'V20_Mixup_Only', 'use_loo': False, 'use_mixup': True},
    {'name': 'V20_LOO_Only', 'use_loo': True, 'use_mixup': False},
    {'name': 'V20_Both', 'use_loo': True, 'use_mixup': True},
]

cv_results = []

for ci, config in enumerate(configs_to_test):
    config_name = config['name']
    use_loo = config['use_loo']
    use_mixup_flag = config['use_mixup']

    print(f'\n  Config #{ci+1}: {config_name} (LOO={use_loo}, Mixup={use_mixup_flag})')

    fold_scores = []
    fold_W = []
    fold_wcos = []
    oof_preds_cfg = np.full_like(deltas, np.nan)

    for fold, (tr_idx, va_idx) in enumerate(all_splits):
        t0 = time.time()
        f_tr_d, f_va_d = deltas[tr_idx], deltas[va_idx]
        f_tr_i, f_va_i = pert_gene_indices[tr_idx], pert_gene_indices[va_idx]
        f_tr_w, f_va_w = weights_array[tr_idx], weights_array[va_idx]
        f_bl = baseline_wmae[va_idx]
        val_pert_set = {pert_names[i] for i in va_idx}
        val_pert_gene_set = {pert_gene_indices[i] for i in va_idx if pert_gene_indices[i] >= 0}
        f_pb_i, f_pb_d, f_pb_w, f_pb_sw = get_fold_pb(val_pert_set)

        # LOO data (exclude val genes to prevent leakage)
        if use_loo:
            f_loo_i, f_loo_d, f_loo_w, f_loo_sw = get_fold_loo(val_pert_gene_set)
        else:
            e = np.float32
            f_loo_i = np.array([], dtype=np.int64)
            f_loo_d = np.zeros((0, cfg.NUM_GENES), dtype=e)
            f_loo_w = np.zeros((0, cfg.NUM_GENES), dtype=e)
            f_loo_sw = np.array([], dtype=e)

        fold_preds = np.zeros((len(va_idx), cfg.NUM_GENES), dtype=np.float32)

        for m_i in range(cfg.N_MLP_ENSEMBLE):
            seed = cfg.SEED + m_i * 7 + fold * 1000
            torch.manual_seed(seed); np.random.seed(seed)
            model = LightMLP(cfg.NUM_GENES, hidden=cfg.MLP_HIDDEN,
                             dropout=cfg.MLP_DROPOUT, depth=cfg.MLP_DEPTH,
                             embed_dim=cfg.EMBED_DIM)
            model, _ = train_v20(
                model, f_pb_i, f_pb_d, f_pb_w, f_pb_sw,
                f_tr_i, f_tr_d, f_tr_w, cfg.GT_UPWEIGHT,
                f_loo_i, f_loo_d, f_loo_w, f_loo_sw,
                f_va_i, f_va_d, f_va_w, f_bl,
                use_mixup=use_mixup_flag,
                mixup_p=cfg.MIXUP_P, mixup_beta=cfg.MIXUP_BETA,
                device=cfg.DEVICE)
            model.eval()
            with torch.no_grad():
                p = model(torch.tensor(f_va_i, dtype=torch.long)).numpy()
            fold_preds += p
            del model; clean_memory()
        fold_preds /= cfg.N_MLP_ENSEMBLE

        fold_preds_ko = np.array([apply_ko(fold_preds[vi], f_va_i[vi], knockout_stats)
                                  for vi in range(len(va_idx))])
        for vi, oi in enumerate(va_idx):
            oof_preds_cfg[oi] = fold_preds_ko[vi]

        s, w, wcos = metric.score(fold_preds_ko, f_va_d, f_va_w, f_bl)
        fold_scores.append(s)
        fold_W.append(w)
        fold_wcos.append(wcos)
        elapsed = time.time() - t0
        print(f'    Fold {fold+1}: {s:.4f} (W={w:.2f}, wcos={wcos:.4f}) [{elapsed/60:.1f} min]')

    cv_mean = np.mean(fold_scores)
    cv_std = np.std(fold_scores)
    oof_score, oof_w, oof_wcos = metric.score(oof_preds_cfg, deltas, weights_array, baseline_wmae)

    cv_results.append({
        'name': config_name, 'config': config,
        'cv_mean': cv_mean, 'cv_std': cv_std,
        'folds': fold_scores, 'fold_W': fold_W, 'fold_wcos': fold_wcos,
        'oof_score': oof_score, 'oof_w': oof_w, 'oof_wcos': oof_wcos,
        'oof_preds': oof_preds_cfg.copy(),
    })
    print(f'  CV: {cv_mean:.4f} +/- {cv_std:.4f} | OOF: {oof_score:.4f}')

# --- Pick winner ---
cv_results.sort(key=lambda x: -x['cv_mean'])

print(f'\n{"="*60}')
print(f'5-FOLD CV RESULTS')
print(f'{"="*60}')
for i, r in enumerate(cv_results):
    marker = ' *** WINNER ***' if i == 0 else ''
    folds_str = '[' + ', '.join([f'{s:.3f}' for s in r['folds']]) + ']'
    print(f'  #{i+1} {r["name"]}:')
    print(f'    CV={r["cv_mean"]:.4f}+/-{r["cv_std"]:.4f}')
    print(f'    OOF={r["oof_score"]:.4f} (W={r["oof_w"]:.2f}, wcos={r["oof_wcos"]:.4f})')
    print(f'    Folds: {folds_str}{marker}')

# Determine winner config
final_winner = cv_results[0]
final_config = final_winner['config']
oof_preds = final_winner['oof_preds']
fold_scores = final_winner['folds']
cv_mean = final_winner['cv_mean']
cv_std = final_winner['cv_std']

# Delta check (baseline is now potentially anywhere in the list)
baseline_result = next(r for r in cv_results if r['name'] == 'V17_baseline')
delta_cv = final_winner['cv_mean'] - baseline_result['cv_mean']
print(f'\nWinner vs Baseline delta: {delta_cv:+.4f}')
if delta_cv < -0.02 and final_winner['name'] != 'V17_baseline':
    print(f'WARNING: Winner is WORSE than V17 baseline. Falling back to V17.')
    final_winner = baseline_result
    final_config = final_winner['config']

# --- Overfit Risk Report ---
print(f'\n{"="*60}')
print(f'OVERFIT RISK REPORT')
print(f'{"="*60}')
cv_ratio = cv_std / (abs(cv_mean) + 1e-8)
status = 'OK' if cv_ratio < 0.3 else 'WARN' if cv_ratio < 0.5 else 'DANGER'
print(f'  Fold variance:    std/mean = {cv_ratio:.3f}  [{status}]')
predicted_lb = cv_mean * 4.14  # V16 had 4.14x ratio
print(f'  Predicted LB:     {predicted_lb:.3f}  (using V16 4.14x ratio)')

# --- Plots ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(range(1, len(fold_scores)+1), fold_scores, color='steelblue', edgecolor='black')
axes[0].axhline(cv_mean, color='red', linestyle='--', label=f'mean={cv_mean:.3f}')
axes[0].set_xlabel('Fold'); axes[0].set_ylabel('Score')
axes[0].set_title(f'5-Fold CV ({final_winner["name"]})'); axes[0].legend()

config_names = [r['name'][:15] for r in cv_results]
config_means = [r['cv_mean'] for r in cv_results]
config_stds = [r['cv_std'] for r in cv_results]
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(cv_results))]
axes[1].bar(range(len(cv_results)), config_means, yerr=config_stds,
            color=colors, edgecolor='black', alpha=0.8, capsize=5)
axes[1].set_xticks(range(len(cv_results)))
axes[1].set_xticklabels(config_names, fontsize=8, rotation=15)
axes[1].set_ylabel('CV Score'); axes[1].set_title('Config Comparison')

terms, wcos_oof = metric.score_per_pert(oof_preds, deltas, weights_array, baseline_wmae)
n_positive = (terms > 0).sum()
axes[2].hist(terms, bins=20, edgecolor='black', alpha=0.7)
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_xlabel('W term'); axes[2].set_ylabel('Count')
axes[2].set_title(f'Per-Pert W ({n_positive}/{len(terms)} positive)')

plt.tight_layout(); plt.show()
print('=' * 70)

In [ ]:
# ============================================================
# CELL 11: Final Training + Submission (V20)
#
# Train on ALL 80 GT + ALL PB + ALL LOO data. N_ENSEMBLE models.
# Uses winning config from Cell 10 CV.
# KO correction ONLY. Rows 61-120 = zeros.
# ============================================================
print('=' * 70)
print(f'CELL 11: FINAL TRAINING + SUBMISSION (V20)')
print(f'  Winner: {final_winner["name"]}')
print(f'  Config: {final_config}')
print(f'  CV={cv_mean:.4f} +/- {cv_std:.4f}')
print('=' * 70)

use_loo_final = final_config.get('use_loo', True)
use_mixup_final = final_config.get('use_mixup', True)

# --- 90/10 early stopping split ---
n_es = max(8, len(pert_names) // 10)
es_idx = np.random.RandomState(cfg.SEED).choice(len(pert_names), size=n_es, replace=False)
tr_idx_final = np.array([i for i in range(len(pert_names)) if i not in es_idx])

tr_d_f, es_d_f = deltas[tr_idx_final], deltas[es_idx]
tr_i_f = pert_gene_indices[tr_idx_final]
es_i_f = pert_gene_indices[es_idx]
tr_w_f, es_w_f = weights_array[tr_idx_final], weights_array[es_idx]
bl_es = baseline_wmae[es_idx]

# PB for final training (exclude ES perts only)
es_pert_set = {pert_names[i] for i in es_idx}
es_gene_set = {pert_gene_indices[i] for i in es_idx if pert_gene_indices[i] >= 0}
pb_i_final, pb_d_final, pb_w_final, pb_sw_final = get_fold_pb(es_pert_set)

# LOO for final training (exclude ES genes only)
if use_loo_final:
    loo_i_final, loo_d_final, loo_w_final, loo_sw_final = get_fold_loo(es_gene_set)
else:
    e = np.float32
    loo_i_final = np.array([], dtype=np.int64)
    loo_d_final = np.zeros((0, cfg.NUM_GENES), dtype=e)
    loo_w_final = np.zeros((0, cfg.NUM_GENES), dtype=e)
    loo_sw_final = np.array([], dtype=e)

print(f'Final training: {len(tr_idx_final)} GT + {len(pb_i_final)} PB + {len(loo_i_final)} LOO')
print(f'ES holdout: {len(es_idx)} perts')

val_gi_list = list(val_gene_indices)

# --- Train and predict ---
print(f'\n--- Training V20 ({cfg.N_MLP_ENSEMBLE} models) ---')
t0 = time.time()
final_preds = np.zeros((len(val_pert_ids), cfg.NUM_GENES), dtype=np.float32)

for m_i in tqdm(range(cfg.N_MLP_ENSEMBLE), desc='V20 training'):
    seed = cfg.SEED + 9999 + m_i * 7
    torch.manual_seed(seed); np.random.seed(seed)
    model = LightMLP(cfg.NUM_GENES, hidden=cfg.MLP_HIDDEN,
                     dropout=cfg.MLP_DROPOUT, depth=cfg.MLP_DEPTH,
                     embed_dim=cfg.EMBED_DIM)
    model, _ = train_v20(
        model, pb_i_final, pb_d_final, pb_w_final, pb_sw_final,
        tr_i_f, tr_d_f, tr_w_f, cfg.GT_UPWEIGHT,
        loo_i_final, loo_d_final, loo_w_final, loo_sw_final,
        es_i_f, es_d_f, es_w_f, bl_es,
        use_mixup=use_mixup_final,
        mixup_p=cfg.MIXUP_P, mixup_beta=cfg.MIXUP_BETA,
        device=cfg.DEVICE)
    model.eval()
    with torch.no_grad():
        p = model(torch.tensor(val_gi_list, dtype=torch.long)).numpy()
    final_preds += p
    del model; clean_memory()

final_preds /= cfg.N_MLP_ENSEMBLE
elapsed = (time.time() - t0) / 60
print(f'Training done: {cfg.N_MLP_ENSEMBLE} models in {elapsed:.1f} min')

# --- KO correction ONLY ---
for pi in range(len(val_pert_ids)):
    gene_idx = val_gi_list[pi] if pi < len(val_gi_list) else -1
    final_preds[pi] = apply_ko(final_preds[pi], gene_idx, knockout_stats)

print(f'Predictions: shape={final_preds.shape}, range=[{final_preds.min():.4f}, {final_preds.max():.4f}]')

# Rows 61-120 should be zeros
if not (final_preds[60:] == 0).all():
    print(f'WARNING: Rows 61-120 not all zeros! Setting to zeros.')
    final_preds[60:] = 0.0
else:
    print(f'Rows 61-120 all zeros: OK')

# --- Build and save submission ---
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
sub = pd.DataFrame(final_preds, columns=gene_names)
sub.insert(0, 'pert_id', val_pert_ids)
assert list(sub.columns) == SUBMISSION_COLUMNS, 'Column mismatch!'
assert len(sub) == len(val_pert_ids), 'Row count mismatch!'
assert not sub.isnull().any().any(), 'NaN found!'

cv_str = f'{cv_mean:.4f}'.replace('.', '_')
sub_path = os.path.join(OUTPUT_DIR, f'submission_v20_loo_{ts}_{cv_str}.csv')
sub.to_csv(sub_path, index=False)
sub_local = f'/content/submission_v20_loo_{ts}.csv'
sub.to_csv(sub_local, index=False)
print(f'\nSaved: {sub_local} ({sub.shape})')

# --- Cross-Version Comparison ---
print(f'\n--- Cross-Version Comparison ---')
prev_files = [f for f in os.listdir(OUTPUT_DIR)
              if f.endswith('.csv') and 'submission' in f.lower() and 'v20' not in f.lower()]
for prev_f in sorted(prev_files)[-4:]:
    try:
        prev_sub = pd.read_csv(os.path.join(OUTPUT_DIR, prev_f))
        prev_preds = prev_sub.iloc[:, 1:].values.astype(np.float32)
        corr_prev = np.corrcoef(final_preds[:60].ravel(), prev_preds[:60].ravel())[0, 1]
        mae_prev = np.mean(np.abs(final_preds[:60] - prev_preds[:60]))
        print(f'  V20 vs {prev_f}: corr={corr_prev:.4f}, MAE={mae_prev:.4f}')
    except Exception as e:
        print(f'  Could not compare with {prev_f}: {e}')

print(f'\nUpload to Kaggle: {sub_local}')
print('=' * 70)

In [ ]:
# ============================================================
# CELL 12: Post-Mortem Template (V20)
# ============================================================
print('=' * 70)
print('V20 COMPLETE -- POST-MORTEM TEMPLATE')
print('=' * 70)
print(f'V20 POST-MORTEM')
print(f'===============')
print(f'LB Score:           [SUBMIT TO CHECK]')
print(f'CV Score:           {cv_mean:.4f} +/- {cv_std:.4f}')
print(f'LB/CV Ratio:        [FILL AFTER LB] (V16 was 4.14x)')
print(f'vs V17 LB (3.63):   [FILL AFTER LB]')
print(f'vs V16 LB (3.58):   [FILL AFTER LB]')
print(f'')
print(f'V20 Config:')
print(f'  Architecture: LightMLP (V17 FROZEN)')
print(f'  Loss: cosine_light (lambda={cfg.COSINE_LAMBDA}, cos_right={cfg.COS_RIGHT})')
print(f'  Booster 1: 3-Tier PB ({len(pb_deltas)} samples)')
print(f'  Booster 2: LOO Synthetic ({len(loo_deltas)} samples, {len(set(loo_gene_indices))} genes, weight={cfg.LOO_SAMPLE_WEIGHT})')
print(f'  Booster 3: Mixup (p={cfg.MIXUP_P}, beta={cfg.MIXUP_BETA})')
print(f'  N_ensemble: {cfg.N_MLP_ENSEMBLE}')
print(f'  Training time: {elapsed:.1f} min')
print(f'')
print(f'5-Fold CV Results:')
for r in cv_results:
    folds_str = '[' + ', '.join([f'{s:.3f}' for s in r['folds']]) + ']'
    print(f'  {r["name"]}: CV={r["cv_mean"]:.4f}+/-{r["cv_std"]:.4f}, OOF={r["oof_score"]:.4f}')
    print(f'    Folds: {folds_str}')
print(f'')
print(f'Predicted LB (4.14x): {predicted_lb:.3f}')
print(f'')
print(f'What worked:')
print(f'  1. [FILL AFTER LB]')
print(f'  2. [FILL AFTER LB]')
print(f'')
print(f'What did not:')
print(f'  1. [FILL AFTER LB]')
print(f'  2. [FILL AFTER LB]')
print(f'')
print(f'If LB > 3.63: LOO/Mixup augmentation broke the V17 ceiling.')
print(f'If LB ~ 3.63: Augmentation is neutral. Architecture IS the ceiling.')
print(f'If LB < 3.63: LOO synthetic data hurts. Quality gating needs work.')
print(f'')
print(f'Upload to Kaggle: {sub_local}')
print('=' * 70)

In [ ]:
# TODO: Delete this cell (leftover from V17)

In [ ]:
# TODO: Delete this cell (leftover from V17)